In [58]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import os 
import rasterio
from rasterio.transform import rowcol
from rasterio.transform import xy
from rasterio.windows import Window

# Data Explanation : 
1. Fauna : all animals distribution (amphibians , birds , mammals , reptiles) the data gives th nombre of birds , mammals ...etc in this area for example each pixel containe 1km²and the value is he number of animals in this area (form 0 --> 153  or 0 --> 178)

2. vegetation : the percentage of a specific land cover  (barren , evergreen, herbaceous) in speicific area (pixel)

3. human activities : gives the human modification in 1km² area between 0 and 1 

we can deduct somethings like : 
- High Species Richness + Low Human Modification → Biodiversity hotspots with low disturbance (ideal for conservation).
- High Species Richness + High Human Modification → Threatened ecosystems (urgent conservation priority).
- Low Species Richness + High Human Modification → Urban/agricultural areas with low ecological value.
- Low Species Richness + Low Human Modification → Natural areas with fewer species (e.g., deserts).

In [59]:
df = pd.read_csv("../results/Soil/full_splined_NAN.csv")
df

,Unnamed: 0,SID,X,Y,UD,LD,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM%,Ranges
0,0,-0.09455116238_35.78682408,-0.094551,35.786824,0.0,5.0,7.110377,6.404642,7.311231,12.333685,18.140976,74.547794,0.893930,2.306617,0.533126,0_5
1,1,-0.09455116238_35.78682408,-0.094551,35.786824,5.0,15.0,7.144755,6.190028,7.048924,11.847969,18.278750,74.672326,0.946038,2.353142,0.467774,5_15
2,2,-0.09455116238_35.78682408,-0.094551,35.786824,15.0,30.0,7.293732,5.262191,5.915067,9.743061,18.874708,75.210226,1.171362,2.554309,0.474109,15_30
3,3,-0.09455116238_35.78682408,-0.094551,35.786824,30.0,60.0,7.591430,3.901625,4.290227,5.525814,19.820767,75.889006,1.512805,2.855176,0.356968,30_60
4,4,-0.09455116238_35.78682408,-0.094551,35.786824,60.0,100.0,7.651464,6.050623,7.109795,4.621229,18.809306,74.080899,1.047307,2.419303,0.262489,60_100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3181,3181,8.414166667_37.09472222,8.414167,37.094722,5.0,15.0,6.177699,28.007083,6.131968,0.100000,24.892460,68.975573,1.401231,2.420257,11.696769,5_15
3182,3182,8.414166667_37.09472222,8.414167,37.094722,15.0,30.0,6.271719,26.751345,6.165818,0.100000,25.601487,68.232695,1.423329,2.403869,14.661547,15_30
3183,3183,8.414166667_37.09472222,8.414167,37.094722,30.0,60.0,6.405402,20.919296,6.210706,0.100000,27.210403,66.578892,1.477367,2.372119,18.866199,30_60
3184,3184,8.414166667_37.09472222,8.414167,37.094722,60.0,100.0,6.314960,19.177498,6.166735,0.100000,28.642714,65.190550,1.535705,2.358143,15.976235,60_100


In [60]:
df = df.drop(columns=["Unnamed: 0" , "SID"])
df

,X,Y,UD,LD,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM%,Ranges
0,-0.094551,35.786824,0.0,5.0,7.110377,6.404642,7.311231,12.333685,18.140976,74.547794,0.893930,2.306617,0.533126,0_5
1,-0.094551,35.786824,5.0,15.0,7.144755,6.190028,7.048924,11.847969,18.278750,74.672326,0.946038,2.353142,0.467774,5_15
2,-0.094551,35.786824,15.0,30.0,7.293732,5.262191,5.915067,9.743061,18.874708,75.210226,1.171362,2.554309,0.474109,15_30
3,-0.094551,35.786824,30.0,60.0,7.591430,3.901625,4.290227,5.525814,19.820767,75.889006,1.512805,2.855176,0.356968,30_60
4,-0.094551,35.786824,60.0,100.0,7.651464,6.050623,7.109795,4.621229,18.809306,74.080899,1.047307,2.419303,0.262489,60_100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3181,8.414167,37.094722,5.0,15.0,6.177699,28.007083,6.131968,0.100000,24.892460,68.975573,1.401231,2.420257,11.696769,5_15
3182,8.414167,37.094722,15.0,30.0,6.271719,26.751345,6.165818,0.100000,25.601487,68.232695,1.423329,2.403869,14.661547,15_30
3183,8.414167,37.094722,30.0,60.0,6.405402,20.919296,6.210706,0.100000,27.210403,66.578892,1.477367,2.372119,18.866199,30_60
3184,8.414167,37.094722,60.0,100.0,6.314960,19.177498,6.166735,0.100000,28.642714,65.190550,1.535705,2.358143,15.976235,60_100


In [61]:
df_unique = df.copy()
df_unique = df_unique.drop_duplicates(subset=["X" ,"Y"])
df_unique 

,X,Y,UD,LD,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM%,Ranges
0,-0.094551,35.786824,0.0,5.0,7.110377,6.404642,7.311231,12.333685,18.140976,74.547794,0.893930,2.306617,0.533126,0_5
6,-0.094682,35.783222,0.0,5.0,7.576187,6.028645,8.795516,4.101079,7.912891,83.291593,0.000000,2.247701,0.567299,0_5
12,-0.094814,35.779620,0.0,5.0,7.557264,5.292950,6.339849,4.232644,23.833469,69.783936,1.329243,2.392225,0.745866,0_5
18,-0.094945,35.776018,0.0,5.0,7.668368,3.400624,3.270652,14.641522,50.034298,46.695050,2.763003,2.696493,0.676091,0_5
24,-0.095076,35.772417,0.0,5.0,7.051867,2.440479,1.517456,9.439977,39.378934,59.103610,3.200028,3.605780,0.653315,0_5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3156,7.698333,36.781111,0.0,5.0,8.299528,383.853774,30.381132,0.099906,38.016981,27.490613,0.224196,0.000000,1.004885,0_5
3162,7.992500,36.848611,0.0,5.0,5.687539,28.007083,4.610997,0.121828,26.872473,68.516530,1.761085,2.698702,48.783917,0_5
3168,8.200000,36.900000,0.0,5.0,5.614928,23.527257,3.171597,0.100015,42.066458,54.761945,2.579962,2.843597,34.722968,0_5
3174,8.386667,37.064444,0.0,5.0,5.094341,27.049788,12.245741,0.100000,24.879419,62.874878,0.708640,1.635620,27.972723,0_5


In [62]:
def latlon_to_pixel(lat, lon, dataset ,transform):
    # Validate if the coordinate is within the image bounds
    print(dataset.bounds.left , lat , dataset.bounds.right)
    print(dataset.bounds.bottom , lon , dataset.bounds.top)

    if not (dataset.bounds.left <= lat <= dataset.bounds.right and
            dataset.bounds.bottom <= lon <= dataset.bounds.top):
        lat -=0.1
        lon -=0.1
    # Convert to pixel row & column
    row, col = rowcol(dataset.transform, lon, lat)
    return row, col

In [63]:
# windows Forming  for full one file (nefthou el file mera brk)
def window_forming(raster_path, df, height=15, width=15 , var=None):
    covariable_files = os.listdir(raster_path)
    covariable_num = len(covariable_files)

    datasets = [rasterio.open(os.path.join(raster_path, fname)) for fname in covariable_files]
    transforms = [ds.transform for ds in datasets]
    print(len(datasets))
    windows = []
    mid = height // 2

    for i in range(len(df)):
        print(f"Processing point {i}")
        lat = df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]
        window = np.empty((height, width, covariable_num))

        for idx, src in enumerate(datasets):
            transform = transforms[idx] 
            pixel_y, pixel_x = latlon_to_pixel(lat, lon, src, transform)
            print(src.crs)
            print(src.res)
            try:
                win = Window(pixel_x - mid, pixel_y - mid, width, height)
                data = src.read(1, window=win)
                window[:, :, idx] = data

            except Exception as e:
                print(f"Error at point {i}, covariable {idx}")
                print(e)
                window[:, :, idx] = np.nan  # Fill with NaNs on failure

        windows.append(window)

    for ds in datasets:
        ds.close()

    return windows

In [64]:
# for devided tiff files 
def window_forming2(raster_path  , df ,height=15 , width=15 ):
    covariable_num = 1
    windows = []

    for i in range(len(df)) :
        print(f"the dataset is in : {i}") 
        lat =  df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]
        window = np.empty((height, width, covariable_num))
        for index , j in enumerate(os.listdir(raster_path)):
            with rasterio.open(f"{raster_path}/{j}") as src :
                raster_data = src.read()
                transform = src.transform 
            try : 
                pixel_x ,pixel_y = latlon_to_pixel(lat , lon , src , transform)
                mid = height//2
                window[:,:,index] = raster_data[ 0 , pixel_x - mid : pixel_x + mid +1 , pixel_y - mid : pixel_y + mid +1 ]
                break 
            except Exception as e:
                print("can'found in ")
                continue 
        windows.append(window)
    return windows

In [65]:
def window_forming3(all_files  , df ,height=15 , width=15 ):
    covariable_num = len(all_files)
    windows = []
    for i in range(len(df)) :
        print(f"the dataset is in : {i}") 
        lat =  df.iloc[i]["X"]
        lon = df.iloc[i]["Y"]

        window = np.empty((height, width, covariable_num))

        for index , i in enumerate(all_files):
            raster_data = i.raster_data
            pixel_x ,pixel_y = latlon_to_pixel(lat , lon ,  i.src , i.transform)
            print(pixel_x , pixel_y)
            mid = height//2
            try : 
                window[:,:,index] = raster_data[ pixel_x - mid : pixel_x + mid +1 , pixel_y - mid : pixel_y + mid +1 ]
            except Exception as e:
                print("error is here ")
                print(e)
        windows.append(window)
    return windows

### 1.Climat

In [66]:
climat_window = window_forming("../Resampled/climat/" , df_unique)
climat_window = np.array(climat_window)
np.save("../Data_multi_CNN/climat.npy", climat_window)

6
Processing point 0
-180.0 -0.0945511623800001 180.0
-90.0 35.78682408 90.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-90.0 35.78682408 90.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-90.0 35.78682408 90.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-90.0 35.78682408 90.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-90.0 35.78682408 90.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-90.0 35.78682408 90.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
Processing point 1
-180.0 -0.0946823495400002 180.0
-90.0 35.78322222 90.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0946823495400002 180.0
-90.0 35.78322222 90.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0946823495400002 180.0
-90.0 35.78322222 90.0
EPSG:4326
(0.008

In [67]:
climat_window.shape

(531, 15, 15, 6)

### 2. organisme

In [68]:
organisme = window_forming("../Resampled/organisme/" , df_unique)
organisme = np.array(organisme)
print(organisme.shape)
np.save("../Data_multi_CNN/organisme.npy", organisme)

20
Processing point 0
-179.999948976711 -0.0945511623800001 179.84350768202054
-89.04039831914453 35.78682408 83.88685421908791
EPSG:4326
(0.008333374786566582, 0.008333441884161363)
-179.999948976711 -0.0945511623800001 179.84350768202054
-89.04039831914453 35.78682408 83.88685421908791
EPSG:4326
(0.008333374786566582, 0.008333441884161363)
-180.0 -0.0945511623800001 179.9985600001673
-90.0 35.78682408 89.99928000008367
EPSG:4326
(0.008333492904932228, 0.008333685818791781)
-179.999948976711 -0.0945511623800001 179.84350768202054
-89.04039831914453 35.78682408 83.88685421908791
EPSG:4326
(0.008333374786566582, 0.008333441884161363)
-179.99999991978757 -0.0945511623800001 179.9971167959036
-60.000802620970376 35.78682408 83.64317286591495
EPSG:4326
(0.008333459494795971, 0.008333467278928197)
-180.0 -0.0945511623800001 179.9999985600001
-90.00000000000006 35.78682408 89.99999928
EPSG:4326
(0.008333333300000002, 0.008333333300000002)
-180.0 -0.0945511623800001 179.9999985600001
-90.0000

### 3. relief

In [69]:
relief = window_forming("../Resampled/relief/" , df_unique)
relief = np.array(relief)
print(relief.shape)
np.save("../Data_multi_CNN/relief.npy", relief)

6
Processing point 0
-180.0 -0.0945511623800001 180.0
-56.0 35.78682408 84.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-56.0 35.78682408 84.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-56.0 35.78682408 84.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-56.0 35.78682408 84.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-56.0 35.78682408 84.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0945511623800001 180.0
-56.0 35.78682408 84.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
Processing point 1
-180.0 -0.0946823495400002 180.0
-56.0 35.78322222 84.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0946823495400002 180.0
-56.0 35.78322222 84.0
EPSG:4326
(0.008333333333333333, 0.008333333333333333)
-180.0 -0.0946823495400002 180.0
-56.0 35.78322222 84.0
EPSG:4326
(0.008

### 4. Spatial Position

In [70]:
landuse_windo = window_forming2("../Resampled/Spatial_position/" , df_unique)
np.array(landuse_windo).shape

the dataset is in : 0
0.0 -0.0945511623800001 10.000000000000002
9.999999999999998 35.78682408 20.0
can'found in 
10.0 -0.0945511623800001 20.0
9.999999999999998 35.78682408 20.0
can'found in 
-10.0 -0.0945511623800001 1.7763568394002505e-15
9.999999999999998 35.78682408 20.0
can'found in 
0.0 -0.0945511623800001 10.000000000000002
20.0 35.78682408 30.0
can'found in 
10.0 -0.0945511623800001 20.0
20.0 35.78682408 30.0
can'found in 
-10.0 -0.0945511623800001 1.7763568394002505e-15
20.0 35.78682408 30.0
can'found in 
-20.0 -0.0945511623800001 -9.999999999999998
20.0 35.78682408 30.0
can'found in 
0.0 -0.0945511623800001 10.000000000000002
30.0 35.78682408 40.0
can'found in 
-10.0 -0.0945511623800001 1.7763568394002505e-15
30.0 35.78682408 40.0
can'found in 
the dataset is in : 1
0.0 -0.0946823495400002 10.000000000000002
9.999999999999998 35.78322222 20.0
can'found in 
10.0 -0.0946823495400002 20.0
9.999999999999998 35.78322222 20.0
can'found in 
-10.0 -0.0946823495400002 1.7763568394002

(531, 15, 15, 1)

In [71]:
def replace_nan_with_column_mean(data):
    data_filled = data.copy()
    for idx in range(data.shape[0]):  # for each matrix
        matrix = data_filled[idx, :, :, 0]  # shape (9, 9)
        nan_indices = np.argwhere(np.isnan(matrix))
        for i, j in nan_indices:
            # Get surrounding row indices (up and down)
            rows_to_consider = []
            if i > 0:
                rows_to_consider.append(matrix[i-1, j])
            if i < 8:
                rows_to_consider.append(matrix[i+1, j])
            # Remove NaNs from consideration
            rows_to_consider = [val for val in rows_to_consider if not np.isnan(val)]
            if rows_to_consider:
                matrix[i, j] = np.mean(rows_to_consider)
            else:
                matrix[i, j] = 0  # or some default if all neighbors are NaN
        data_filled[idx, :, :, 0] = matrix
    return data_filled

landuse_windo = replace_nan_with_column_mean(np.array(landuse_windo))

In [72]:
np.isnan(landuse_windo).sum()

0

In [73]:
np.save("../Data_multi_CNN/landuse.npy" , landuse_windo)

### 4. Parent Material

In [74]:
parent_window = window_forming("../Resampled/Parent_Material/", df_unique)
np.save("../Data_multi_CNN/Parent_Material.npy", parent_window)
np.array(parent_window).shape

1
Processing point 0
-22.4133525492225 -0.0945511623800001 64.54036521609329
-38.574875710885514 35.78682408 40.0038669850103
EPSG:4326
(0.008333689645899539, 0.008333730267885864)
Processing point 1
-22.4133525492225 -0.0946823495400002 64.54036521609329
-38.574875710885514 35.78322222 40.0038669850103
EPSG:4326
(0.008333689645899539, 0.008333730267885864)
Processing point 2
-22.4133525492225 -0.0948135134200001 64.54036521609329
-38.574875710885514 35.77962036 40.0038669850103
EPSG:4326
(0.008333689645899539, 0.008333730267885864)
Processing point 3
-22.4133525492225 -0.0949446540399998 64.54036521609329
-38.574875710885514 35.7760185 40.0038669850103
EPSG:4326
(0.008333689645899539, 0.008333730267885864)
Processing point 4
-22.4133525492225 -0.0950757713799999 64.54036521609329
-38.574875710885514 35.77241663 40.0038669850103
EPSG:4326
(0.008333689645899539, 0.008333730267885864)
Processing point 5
-22.4133525492225 -0.0952409953400001 64.54036521609329
-38.574875710885514 35.768788

(531, 15, 15, 1)

In [ ]:
parent_window

In [ ]:
# shape file transformation
import geopandas as gpd
from rasterio.transform import from_bounds
from rasterio.features import rasterize

def transform_shape(shapefile_path , attribute , output_path):
    gdf = gpd.read_file(shapefile_path)
    gdf["category_numeric"], category_mapping = gdf[attribute].factorize()
    bounds = gdf.total_bounds  
    print("Bounds:", bounds)
    resolution = 0.008983  # -- 1KM
    width = int((bounds[2] - bounds[0]) / resolution)
    height = int((bounds[3] - bounds[1]) / resolution)

    print(f"Raster size: Width={width}, Height={height}")

    transform = from_bounds(*bounds, width, height)
    raster = rasterize(
        [(geom, value) for geom, value in zip(gdf.geometry, gdf["category_numeric"])],
        out_shape=(height, width),
        transform=transform,
        fill=-1, 
        dtype=rasterio.int32
    )

    tiff_path = output_path
    with rasterio.open(
        tiff_path, "w", 
        driver="GTiff",
        height=height, 
        width=width, 
        count=1,
        dtype=rasterio.uint8,
        crs=gdf.crs,  # Use same CRS as shapefile
        transform=transform
    ) as dst:
        dst.write(raster, 1)
    print(f"Saved 1 km raster to {tiff_path}")

In [1]:
transform_shape("../Maps/ParentMaterial/Geomorphology/Algeria/Algeria_HG.shp" , "AlgGLG"  , "output_1km.tif")
geoMorpho_window = window_forming2("../Maps/ParentMaterial/Geomorphology/tiff" , df_unique)
np.save("../Data_multi_CNN/geoMorpho.npy", geoMorpho_window)
geoMorpho_window

NameError: name 'transform_shape' is not defined

In [ ]:
class optimizerKhefoufi:
    def __init__(self , raster_data , transforme , src ):
        self.raster_data = raster_data
        self.transform = transforme
        self.src = src

def data_saver(path):
    files = []
    lat_min = 0 
    lat_max = 10

    lon_min = 30
    lon_max = 40


    for i in os.listdir(path):
        print(i)
        with rasterio.open(f"{path}/{i}") as src :
            raster_data = src.read()
            transform = src.transform 

            pixel_minx , pixel_miny = latlon_to_pixel(lat_min , lon_min ,src, transform)
            pixel_maxx , pixel_maxy = latlon_to_pixel(lat_max , lon_max , src , transform)
            print(pixel_minx , pixel_maxx , pixel_miny , pixel_maxy)
            
            raster_data = raster_data[0 , pixel_minx : pixel_maxx , pixel_miny : pixel_maxy]

            tiff_val = optimizerKhefoufi(raster_data , transform , src)
        files.append(tiff_val)
    return files

all_files = data_saver("../Maps/organisme/Microbial/SoilResp_HeterotrophicResp_1928/data")

FileNotFoundError: [WinError 3] The system cannot find the path specified: '../Maps/organisme/Microbial/SoilResp_HeterotrophicResp_1928/data'

In [ ]:
Microbial_window = window_forming3(all_files , df_unique)
np.save("../Data_multi_CNN/microbial.npy", Microbial_window)
Microbial_window

In [ ]:
def pixel_to_latlon(row, col, dataset):
    lon, lat = xy(dataset.transform, row, col)
    return lat, lon

In [ ]:
windows = []
raster_data_used = "../Resampled/climat/wc2.1_30s_prec_01.tif"
with rasterio.open(raster_data_used) as src: 
    raster = src.read()
    transform = src.transform
    for index in range(len(df_unique)):
        lat = df_unique.iloc[index]["X"]
        lon = df_unique.iloc[index]["Y"]

        pixel_row, pixel_col = latlon_to_pixel(lat, lon, src, transform)
        window = np.zeros((9, 9, 2))  # Only initialize once

        for i in range(pixel_row - 4, pixel_row + 5):
            for j in range(pixel_col - 4, pixel_col + 5):
                lat_val, lon_val = pixel_to_latlon(i, j, src)

                row_idx = i - (pixel_row - 4)
                col_idx = j - (pixel_col - 4)

                window[row_idx, col_idx, 0] = lat_val
                window[row_idx, col_idx, 1] = lon_val

        windows.append(window)

# Check result
print(windows[0].shape)
np.save("../Data_multi_CNN/latlon.npy", windows)

-180.0 -0.0945511623800001 180.0
-90.0 35.78682408 90.0
-180.0 -0.0946823495400002 180.0
-90.0 35.78322222 90.0
-180.0 -0.0948135134200001 180.0
-90.0 35.77962036 90.0
-180.0 -0.0949446540399998 180.0
-90.0 35.7760185 90.0
-180.0 -0.0950757713799999 180.0
-90.0 35.77241663 90.0
-180.0 -0.0952409953400001 180.0
-90.0 35.76878855 90.0
-180.0 -0.0989717369199998 180.0
-90.0 35.7869309 90.0
-180.0 -0.0991027251599998 180.0
-90.0 35.78332903 90.0
-180.0 -0.0992336901700001 180.0
-90.0 35.77972715 90.0
-180.0 -0.0993646319399998 180.0
-90.0 35.77612528 90.0
-180.0 -0.103523124 180.0
-90.0 35.78343567 90.0
-180.0 -0.1036538900999999 180.0
-90.0 35.77983378 90.0
-180.0 -0.1037846331 180.0
-90.0 35.77623189 90.0
-180.0 -0.1080741132999998 180.0
-90.0 35.77994025 90.0
-180.0 -0.1082046573999999 180.0
-90.0 35.77633834 90.0
-180.0 -0.421419637 180.0
-90.0 35.0443358 90.0
-180.0 -0.4215329289 180.0
-90.0 35.04073251 90.0
-180.0 -0.4346766278 180.0
-90.0 35.0410113 90.0
-180.0 -0.4347893238 180.0
-

In [ ]:
windows

[array([[[ 1.33324104, 36.18646279],
         [ 1.33324104, 36.19115959],
         [ 1.33324104, 36.19585638],
         [ 1.33324104, 36.20055318],
         [ 1.33324104, 36.20524997],
         [ 1.33324104, 36.20994677],
         [ 1.33324104, 36.21464357]],
 
        [[ 1.32874942, 36.18646279],
         [ 1.32874942, 36.19115959],
         [ 1.32874942, 36.19585638],
         [ 1.32874942, 36.20055318],
         [ 1.32874942, 36.20524997],
         [ 1.32874942, 36.20994677],
         [ 1.32874942, 36.21464357]],
 
        [[ 1.32425781, 36.18646279],
         [ 1.32425781, 36.19115959],
         [ 1.32425781, 36.19585638],
         [ 1.32425781, 36.20055318],
         [ 1.32425781, 36.20524997],
         [ 1.32425781, 36.20994677],
         [ 1.32425781, 36.21464357]],
 
        [[ 1.31976619, 36.18646279],
         [ 1.31976619, 36.19115959],
         [ 1.31976619, 36.19585638],
         [ 1.31976619, 36.20055318],
         [ 1.31976619, 36.20524997],
         [ 1.31976619, 36.209

In [ ]:
transform_shape("../Maps/ParentMaterial/Rock_type/geo7_2ag.shp" , "GLG"  , "../Maps/ParentMaterial/Rock_type/tiff/rocksType_1km.tif")
rocks_window = window_forming("../Maps/ParentMaterial/Rock_type/tiff", df_unique)
np.save("../Data_multi_CNN/rocks.npy", rocks_window)
rocks_window

,AREA,PERIMETER,GEO2_7G_,GEO2_7G_ID,GLG,geometry,category_numeric
0,2250.44404,966.65344,2,11939,SEA,"POLYGON ((5.40701 5.57146, 5.4109 5.4694, 5.42...",0
1,1.60529,7.58497,3,6,oth,"POLYGON ((15.64759 40.00266, 15.66 40, 15.676 ...",1
2,7.02790,33.05255,4,3,oth,"POLYGON ((19.89965 39.96591, 22.81549 39.8342,...",1
3,0.06592,1.66218,5,7,oth,"POLYGON ((19.67681 39.78778, 19.7106 39.7867, ...",1
4,0.90818,4.50544,6,4,oth,"POLYGON ((8.40944 39.76892, 9.63446 39.84441, ...",1
...,...,...,...,...,...,...,...
11972,0.01677,0.69987,11974,11935,SO,"POLYGON ((19.69988 -34.58595, 19.70456 -34.581...",31
11973,0.00456,0.35706,11975,11936,D,"POLYGON ((19.69988 -34.58595, 19.70059 -34.602...",27
11974,0.00239,0.20456,11976,11934,pCm,"POLYGON ((19.43258 -34.60229, 19.40437 -34.627...",11
11975,0.00641,0.35766,11977,11938,SO,"POLYGON ((19.84761 -34.67162, 19.85932 -34.661...",31


In [ ]:
def outputs_building(df ,df_unique, atr):
    depth_ranges = [
        (0, 15),
        (5, 15),
        (15, 30),
        (30, 60),
        (60, 100),
        (100, 200)
    ]
    result_array = np.zeros((531, 6))
    depth_indices = {i: [] for i in range(len(depth_ranges))}

    for i, (lb, ub) in enumerate(depth_ranges):
        for j in range(lb, ub + 1):
            depth_indices[i].append(j)
    
    for _, row in df.iterrows():
        x, y, ud, ld, val = row['X'], row['Y'], row['UD'], row['LD'] , row[atr]
        for index, depth_range in depth_indices.items():
            if any(d in range(int(ud), int(ld) + 1) for d in depth_range):
                result_array[df_unique[(df_unique['X'] == x) & (df_unique['Y'] == y)].index[0], index] = val

    mean = df[atr].mean()
    result_array = np.nan_to_num(result_array, nan=mean)
    finals = np.array(result_array).reshape(531,6,1)
    for i in finals:
        print(i)
    
    print(finals.shape)
    return finals

In [ ]:
df_unique = df_unique.reset_index()
df_unique

,index,X,Y,UD,LD,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM%,Ranges
0,0,-0.094551,35.786824,0.0,5.0,7.110377,6.404642,7.311231,12.333685,18.140976,74.547794,0.893930,2.306617,0.533126,0_5
1,1,-0.094682,35.783222,0.0,5.0,7.576187,6.028645,8.795516,4.101079,7.912891,83.291593,0.000000,2.247701,0.567299,0_5
2,2,-0.094814,35.779620,0.0,5.0,7.557264,5.292950,6.339849,4.232644,23.833469,69.783936,1.329243,2.392225,0.745866,0_5
3,3,-0.094945,35.776018,0.0,5.0,7.668368,3.400624,3.270652,14.641522,50.034298,46.695050,2.763003,2.696493,0.676091,0_5
4,4,-0.095076,35.772417,0.0,5.0,7.051867,2.440479,1.517456,9.439977,39.378934,59.103610,3.200028,3.605780,0.653315,0_5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
526,526,7.698333,36.781111,0.0,5.0,8.299528,383.853774,30.381132,0.099906,38.016981,27.490613,0.224196,0.000000,1.004885,0_5
527,527,7.992500,36.848611,0.0,5.0,5.687539,28.007083,4.610997,0.121828,26.872473,68.516530,1.761085,2.698702,48.783917,0_5
528,528,8.200000,36.900000,0.0,5.0,5.614928,23.527257,3.171597,0.100015,42.066458,54.761945,2.579962,2.843597,34.722968,0_5
529,529,8.386667,37.064444,0.0,5.0,5.094341,27.049788,12.245741,0.100000,24.879419,62.874878,0.708640,1.635620,27.972723,0_5


In [ ]:
df_unique = df_unique.drop(columns=["index"])
df_unique

,X,Y,UD,LD,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM%,Ranges
0,-0.094551,35.786824,0.0,5.0,7.110377,6.404642,7.311231,12.333685,18.140976,74.547794,0.893930,2.306617,0.533126,0_5
1,-0.094682,35.783222,0.0,5.0,7.576187,6.028645,8.795516,4.101079,7.912891,83.291593,0.000000,2.247701,0.567299,0_5
2,-0.094814,35.779620,0.0,5.0,7.557264,5.292950,6.339849,4.232644,23.833469,69.783936,1.329243,2.392225,0.745866,0_5
3,-0.094945,35.776018,0.0,5.0,7.668368,3.400624,3.270652,14.641522,50.034298,46.695050,2.763003,2.696493,0.676091,0_5
4,-0.095076,35.772417,0.0,5.0,7.051867,2.440479,1.517456,9.439977,39.378934,59.103610,3.200028,3.605780,0.653315,0_5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
526,7.698333,36.781111,0.0,5.0,8.299528,383.853774,30.381132,0.099906,38.016981,27.490613,0.224196,0.000000,1.004885,0_5
527,7.992500,36.848611,0.0,5.0,5.687539,28.007083,4.610997,0.121828,26.872473,68.516530,1.761085,2.698702,48.783917,0_5
528,8.200000,36.900000,0.0,5.0,5.614928,23.527257,3.171597,0.100015,42.066458,54.761945,2.579962,2.843597,34.722968,0_5
529,8.386667,37.064444,0.0,5.0,5.094341,27.049788,12.245741,0.100000,24.879419,62.874878,0.708640,1.635620,27.972723,0_5


In [ ]:
df

,X,Y,UD,LD,pH,CE,CEC,A%,L%,S%,ALR(L),ALR(S),OM%,Ranges
0,-0.094551,35.786824,0.0,5.0,7.110377,6.404642,7.311231,12.333685,18.140976,74.547794,0.893930,2.306617,0.533126,0_5
1,-0.094551,35.786824,5.0,15.0,7.144755,6.190028,7.048924,11.847969,18.278750,74.672326,0.946038,2.353142,0.467774,5_15
2,-0.094551,35.786824,15.0,30.0,7.293732,5.262191,5.915067,9.743061,18.874708,75.210226,1.171362,2.554309,0.474109,15_30
3,-0.094551,35.786824,30.0,60.0,7.591430,3.901625,4.290227,5.525814,19.820767,75.889006,1.512805,2.855176,0.356968,30_60
4,-0.094551,35.786824,60.0,100.0,7.651464,6.050623,7.109795,4.621229,18.809306,74.080899,1.047307,2.419303,0.262489,60_100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3181,8.414167,37.094722,5.0,15.0,6.177699,28.007083,6.131968,0.100000,24.892460,68.975573,1.401231,2.420257,11.696769,5_15
3182,8.414167,37.094722,15.0,30.0,6.271719,26.751345,6.165818,0.100000,25.601487,68.232695,1.423329,2.403869,14.661547,15_30
3183,8.414167,37.094722,30.0,60.0,6.405402,20.919296,6.210706,0.100000,27.210403,66.578892,1.477367,2.372119,18.866199,30_60
3184,8.414167,37.094722,60.0,100.0,6.314960,19.177498,6.166735,0.100000,28.642714,65.190550,1.535705,2.358143,15.976235,60_100


In [ ]:
for i in df.columns[4:-1]:
    print(i)
    finals = outputs_building(df ,df_unique, i)
    np.save(f'../Data_multi_CNN/output{i}.npy', np.array(finals))

pH
[[7.29373154]
 [7.29373154]
 [7.59143019]
 [7.65146429]
 [7.57452488]
 [7.57452488]]
[[7.58622173]
 [7.58622173]
 [7.60088013]
 [7.60088648]
 [7.59851575]
 [7.59851575]]
[[7.66997854]
 [7.66997854]
 [7.89051155]
 [8.11582066]
 [8.21838373]
 [8.21838373]]
[[7.67130487]
 [7.67130487]
 [7.67487779]
 [7.6563084 ]
 [7.62556016]
 [7.62556016]]
[[7.01251675]
 [7.01251675]
 [6.97049256]
 [7.26538293]
 [7.73813716]
 [7.73813716]]
[[7.25033398]
 [7.25033398]
 [7.61246254]
 [7.6451182 ]
 [7.60468725]
 [7.60468725]]
[[7.68976514]
 [7.68976514]
 [7.78616511]
 [8.1666771 ]
 [8.36276623]
 [8.36276623]]
[[7.79399611]
 [7.79399611]
 [8.09441163]
 [8.23945213]
 [8.18851021]
 [8.18851021]]
[[7.60929194]
 [7.60929194]
 [7.69755855]
 [7.98712567]
 [8.03154272]
 [8.03154272]]
[[7.72736942]
 [7.72736942]
 [7.72165181]
 [7.77229999]
 [7.89785135]
 [7.89785135]]
[[7.68617442]
 [7.68617442]
 [7.68698331]
 [7.83417455]
 [8.1644042 ]
 [8.1644042 ]]
[[7.74655942]
 [7.74655942]
 [8.09349752]
 [8.11900094]
 [8.15